# Tutorial 2D rz #
This notebook show how to create initial conditions, run and analyse the simulation of a 2D cylindrical Sedov Taylor hydrodynamics expansion.<br />The executable (Castro2d* should be compiled in the ../build folder with the 1 temperature model EOS_DIR:= gamma_law__ )

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.constants import m_p, e
import sys
from analysis_tool import CastroSimulation
import re
import glob
import subprocess
sys.path.append("../../initial_condition")
from ionization_routines import save_to_openpmd

### Generate inputs for the simulation ###

In [ ]:
# Create simulation folder
sim_folder = '../run/run_tutorial_2d/'
os.makedirs(sim_folder, exist_ok=True)

Define grid and fields initial profile

In [ ]:
r = np.linspace(0.0, 1e-2, 100)
z = np.linspace(0.0, 5e-2, 100)

R, Z = np.meshgrid(r, z, indexing='ij')
width = 50e-6 # width of the Gaussian profile
T_max = 1000.0  # eV
T_eV = T_max * np.exp(- R**2 / width**2) # Gaussian profile of temperature
ionization_fraction = 1. # Define the fraction of ionized hydrogen H1

Save the initial file into OpenPMD inside the simulation folder

In [ ]:
# Species keys from species.net
with open('../build/species.net', 'r') as f:
    species_keys = re.findall(r'\n\s.*\s([A-Z][a-z]*\d)', f.read())

# Populations array
populations = np.zeros((R.shape[0], Z.shape[1], len(species_keys)))
populations[:, :,  species_keys.index('H1')] = ionization_fraction

# Save file to openPMD
save_to_openpmd({'r': [r.min(), r.max()], 'z': [z.min(), z.max()]},
            populations, T_eV, f'{sim_folder}2d_inputs.h5', species_keys)

### Run simulation ###
Define the run simulation function

In [ ]:
def run_castro_simulation(model='gamma_law', runtime_options=''):
    """
    Run the Castro simulation directly (without srun) in a specific folder.
    """
    # Find the Castro executable (absolute path)
    build_dir = os.path.abspath("../build")
    executables = glob.glob(os.path.join(build_dir, f"Castro2d*.{model}.ex"))
    if len(executables) == 0:
        raise FileNotFoundError(f"No Castro2d executable found in {build_dir}")
    elif len(executables) > 1:
        raise RuntimeError(f"Multiple Castro2d executables found: {executables}")
    executable = os.path.abspath(executables[0])

    # Absolute path for the inputs file
    inputs = os.path.abspath("../run/inputs.2d.cyl")

    # Create the simulation folder if it doesn't exist
    os.makedirs(sim_folder, exist_ok=True)

    # Build the command
    command = f"{executable} {inputs} {runtime_options}"

    # Run Castro inside sim_folder
    try:
        subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            check=True,
            cwd=sim_folder
        )
    except subprocess.CalledProcessError as e:
        print(f"Command failed with exit code {e.returncode}")
        print("STDOUT:", e.stdout)
        print("STDERR:", e.stderr)
        raise


Run the simulation with 

In [ ]:
runtime_options=f"geometry.prob_hi = 0.1 0.5 castro.add_ext_src = 0 amr.max_level = 2 castro.diffuse_temp = 0 amr.max_grid_size = 128 problem.initial_conditions_file=2d_inputs.h5"
run_castro_simulation(model='gamma_law', runtime_options=runtime_options) # The simulation take approximately 30 seconds to run

### Analysis ###

Open the simulation folder and the simulation timestep folder named plt_2d_*

In [ ]:
run_dir = sim_folder
file_start = 'plt_2d_' # Prefix for the Castro plotfiles

cs = CastroSimulation(run_dir, file_start)
cs.sim_info() # Display simulation information (domain, grid, time, list of fields, etc.)
max_level = cs.max_level # Get the maximum AMR level, refinement goes from 0 to max_level

### Plot 2D fields ###

In [ ]:
t = 9.0e-9 # time in seconds
m = cs.get_field(t, quantity='density', level=max_level)

plt.figure(figsize=(10,6))
plt.imshow(m['q'], extent=[m['z'][0], m['z'][-1], m['r'][0], m['r'][-1]], origin='lower')
plt.colorbar(shrink = 0.2)
plt.title('2D Density Profile')
plt.xlabel('z')
plt.ylabel('r')

### Get 1D slice of the 2D field ###

In [ ]:
t = 2.0e-9 # time in seconds
m_slice = cs.get_field(t, quantity='density', level=max_level, positions={'z':0.03})

plt.figure(figsize=(6,4))
plt.plot(m_slice['r'], m_slice['q'])
plt.title('Density Profile at z=0.03 cm')
plt.xlabel('r (cm)')
plt.ylabel('Density (g/cm^3)')

### Get energy for a given time ###

In [ ]:
t_single = 2.0e-9 # time in seconds
energy_single = cs.get_energy(t_single, level=max_level)[0]
print(f"Total energy at time {t_single:.2e} s: {energy_single:.2e} erg")

### Get energy for a time range ###

In [ ]:
plt.style.use('default')
t_array = cs.output_times # Get the array of output times
energy_array = cs.get_energy(t_array, level=max_level)[0] # Compute total energy array

thermal_energy_array = cs.get_energy(t_array, level=max_level, energy_type='thermal')[0] # Compute thermal energy array
kinetic_energy_array = cs.get_energy(t_array, level=max_level, energy_type='kinetic')[0] # Compute kinetic energy array

plt.figure()
plt.plot(t_array, thermal_energy_array, label='Thermal Energy')
plt.plot(t_array, kinetic_energy_array, label='Kinetic Energy')
plt.plot(t_array, energy_array, label='Total Energy', linestyle='--', color='black')
plt.xlabel('Time (s)')
plt.ylabel('Total Energy (erg)')
plt.ylim(0, max(energy_array)*1.1)
plt.legend()

### Get the number of particles for a given time ###

In [ ]:
t_single = 2.0e-9 # time in seconds
part_H1_single = cs.get_particle_number(t_single, species='H1', level=max_level)[0]

print(f"Number of H1 particles at time {t_single:.2e} s: {part_H1_single:.2e}")

### Get the particles number for a time array ###

In [ ]:
t_array = cs.output_times
part_H1_array = cs.get_particle_number(t_array, species='H1', level=2)[0]

plt.figure()
plt.plot(t_array, part_H1_array, label='H1 Particle Number')
plt.xlabel('Time (s)')
plt.ylabel('Particle Number')
plt.yscale('log')
plt.legend()